In [ ]:
install.packages("pacman")
if (!requireNamespace("BiocManager", quietly = TRUE)) {
    install.packages("BiocManager")
}
install.packages('devtools')
suppressPackageStartupMessages(library(devtools))
install.packages('R.utils')
suppressPackageStartupMessages(library(R.utils))
install_github("jpvert/tigress")
suppressPackageStartupMessages(library(tigress))
suppressPackageStartupMessages(library(pacman))
suppressPackageStartupMessages(p_load(foreach))
suppressPackageStartupMessages(p_load(parallel))
suppressPackageStartupMessages(p_load(data.table))
suppressPackageStartupMessages(p_load(plyr))
suppressPackageStartupMessages(p_load(dplyr))
suppressPackageStartupMessages(p_load(docstring))
suppressPackageStartupMessages(p_load(decoupleR))
suppressPackageStartupMessages(p_load(this.path))
suppressPackageStartupMessages(p_load(ggpubr))
suppressPackageStartupMessages(p_load(reshape2))
suppressPackageStartupMessages(p_load(data.table))
suppressPackageStartupMessages(p_load(here))
suppressPackageStartupMessages(p_load(tidyverse))


In [ ]:
root <- dirname(getwd())
folder_name <- "TIGRESS"

# Create the folder if it doesn't already exist
if (!dir.exists(folder_name)) {
  dir.create(folder_name)
}

# folder to store genie3 outputs
tigress_path <- file.path(getwd(), folder_name)

resources.dir <- paste0(root, "/resources/") # Adjusted to look for resources folder one level up from the current working directory, since it's a sibling to the R folder, not inside it.
sc.metadata.file <- paste0(resources.dir, "CHOOSE-sc-wt-and-tf-metadata.csv")
sc.data.file <- paste0(resources.dir, "CHOOSE-sc-wt-and-tf-log-norm.csv.gz")
tfs.to.analyze.file <- paste0(resources.dir, "CHOOSE-tf-to-analyze-metadata.csv")

plot.dir <- paste0(root, "/plots/TIGRESS_plot/")
dir.create(plot.dir, showWarnings = FALSE)

root
resources.dir

In [ ]:
# Read in CHOOSE scRNA-seq data and metadata
fread_with_rownames <- function(file) {
  tbl <- as.data.frame(fread(file))
  rownames(tbl) <- tbl$V1
  tbl <- tbl[, !(colnames(tbl) == "V1")]
  tbl
}

In [ ]:
main.data.file <- paste0(resources.dir, "CHOOSE-multiome-wt-log-norm.csv.gz")
main.meta.file <- paste0(resources.dir, "CHOOSE-multiome-wt-metadata.csv")
tfs.to.analyze.file <- paste0(resources.dir, "CHOOSE-tf-to-analyze-metadata.csv")

In [ ]:
# load in the main data matrix 
cat ("reading main data in and printing parts of it")
main.data <- read.csv(main.data.file, row.names = 1, check.names = FALSE)
dim (main.data)
main.data[1:8, 1:5]

#load in the main data's metadata
cat ("reading main data's metadata in and printing parts of it")
main.meta <- read.csv(main.meta.file, row.names = 1, check.names = FALSE)
dim (main.meta)
main.meta[1:8, 1:5]

# load tfs used for testing and clean the duplicate
tf.list <- read.csv(tfs.to.analyze.file)
tf.list 

In [ ]:
# Ensure that the columns of main and the rows of meta match

# fix gene symbols having periods instead of hyphens (genes are rownames)
rownames(main.data) <- gsub("\\.", "-", rownames(main.data))
rownames(main.data)[1:10]
length(rownames(main.data))

# add an identifier to cell data because meta didnt have one
main.meta$cell_id <- rownames(main.meta)
main.meta$cell_id[1:10]
length(main.meta$cell_id)

# get the common cells (cell ids are in main.data columns)
common_cells <- intersect(colnames(main.data), main.meta$cell_id)
common_cells[1:10]
length(common_cells)

# align metadata row order to main.data columns
main.meta <- main.meta[match(colnames(main.data), main.meta$cell_id), ]
dim(main.meta)

# main.meta <- main.meta[, 1:5]
# main.data <- main.data[, 1:5]

# main.meta[1:8, 1:5]
# main.data[1:8, 1:5]

In [ ]:
# Actually start running TIGRESS on the data (Given data takes about 2 hours for 3 TFs)

# get the list of cell types
cell.types <- unique(tf.list$celltype_jf)
tigress.out <- list()
counter <- 1

# Clean TF names once
all.tfs <- unique(trimws(as.character(tf.list[[1]])))
all.tfs <- all.tfs[!is.na(all.tfs) & nzchar(all.tfs)]

# Keep this as FALSE for full runs; set TRUE for quick truncated testing
use.truncated <- FALSE
n.truncated.genes <- 200

for (ct in cell.types) {
    # get the list of cells to test, and make the data subset (genes x cells)
    tested.cells <- main.meta$cell_id[main.meta$celltype_jf == ct]
    tested.cells <- intersect(tested.cells, colnames(main.data))

    if (length(tested.cells) < 2) {
        warning(paste("Skipping", ct, "- fewer than 2 matched cells."))
        next
    }

    data.sub <- main.data[, tested.cells, drop = FALSE]

    # Optional truncation for quick tests: keep first N genes + all TF genes
    if (use.truncated) {
        keep.genes <- unique(c(rownames(main.data)[1:n.truncated.genes], all.tfs))
        keep.genes <- intersect(keep.genes, rownames(main.data))
        data.sub <- data.sub[keep.genes, , drop = FALSE]
    }

    # Remove genes with 0 variance across selected cells
    data.sub <- data.sub[apply(data.sub, 1, sd) > 0, , drop = FALSE]

    # TIGRESS expects rows = samples/cells and columns = genes
    expdata <- t(data.sub)

    # TFs must be present in colnames(expdata) for TIGRESS
    tfs <- intersect(all.tfs, colnames(expdata))
    missing.tfs <- setdiff(all.tfs, colnames(expdata))

    if (length(tfs) < 2) {
        warning(paste(
            "Skipping", ct,
            "- fewer than 2 TFs available. Missing TFs (first 10):",
            paste(head(missing.tfs, 10), collapse = ", ")
        ))
        next
    }

    cat(
        "Cell type:", ct,
        "| dim(expdata):", paste(dim(expdata), collapse = " x "),
        "| TFs:", length(tfs),
        "| Missing TFs:", length(missing.tfs),
        "\n"
    )

    # add TIGRESS output and increment counter
    tigress.out[[counter]] <- tigress(expdata, tfs, allsteps = FALSE, nstepsLARS = min(length(tfs) - 1, 50))
    names(tigress.out)[counter] <- ct
    counter <- counter + 1
}

tigress.out

In [ ]:
# Write each of the outputs to a file. This was done for storage, so I didnt have to rerun TIGRESS every single time. 

write.csv(tigress.out[1], file = file.path(work.dir, "ge.in.grn.csv.gz"))
write.csv(tigress.out[2], file = file.path(work.dir, "ctx.npc.grn.csv.gz"))
write.csv(tigress.out[3], file = file.path(work.dir, "ctx.ip.grn.csv.gz"))

In [ ]:
ge.in.data <- read.csv("ge.in.grn.csv.gz", row.names = 1)
ctx.npc.data <- read.csv("ctx.npc.grn.csv.gz", row.names = 1)
ctx.ip.data <- read.csv("ctx.ip.grn.csv.gz", row.names = 1)

tigress.list <- list(
    ge_in = ge.in.data,
    ctx_npc = ctx.npc.data,
    ctx_ip = ctx.ip.data
)

In [ ]:
rownames(ge.in.data)
rownames(ctx.npc.data)
rownames(ctx.ip.data)

tigress.list

In [ ]:
main.data <- t(main.data)

In [ ]:
# Postprocess TIGRESS outputs analogously to postprocess-CHOOSE-grnboost.Rmd
library(dplyr)
library(tidyr)
library(plyr)

# Helper: TIGRESS can return either a matrix/data.frame or a list of matrices (allsteps=TRUE).
extract_tigress_matrix <- function(x) {
  if (is.matrix(x) || is.data.frame(x)) {
    return(as.data.frame(x))
  }

  if (is.list(x) && length(x) > 0) {
    # If allsteps=TRUE, use the last LARS step matrix.
    last_obj <- x[[length(x)]]
    if (is.matrix(last_obj) || is.data.frame(last_obj)) {
      return(as.data.frame(last_obj))
    }
  }

  stop("Unsupported TIGRESS output format for one cell type.")
}

# Build a list of TIGRESS matrices by cell type.
tigress.by.ct <- list()

if (exists("tigress.out") && length(tigress.out) > 0) {
  ct.names <- names(tigress.out)
  if (is.null(ct.names)) {
    ct.names <- rep("unknown_celltype", length(tigress.out))
  }

  for (i in seq_along(tigress.out)) {
    mat <- extract_tigress_matrix(tigress.out[[i]])
    tigress.by.ct[[ct.names[i]]] <- mat
  }
} else {
  # Fallback: read .grn outputs from disk if in-memory TIGRESS output is absent.
  grn.files <- list.files(path = tigress_path, pattern = "\\.grn\\.csv\\.gz$", full.names = TRUE)
  if (length(grn.files) == 0) {
    grn.files <- list.files(pattern = "\\.grn\\.csv\\.gz$", full.names = TRUE)
  }

  for (file in grn.files) {
    ct <- sub("\\.grn\\.csv\\.gz$", "", basename(file))
    tigress.by.ct[[ct]] <- as.data.frame(read.csv(file, row.names = 1, check.names = FALSE))
  }
}

if (length(tigress.by.ct) == 0) {
  stop("No TIGRESS outputs found in memory (tigress.out) or as .grn.csv.gz files.")
}

# Ensure expression matrix orientation is genes x cells for correlation.
expr.mat <- main.data
all.tfs <- unique(trimws(as.character(tf.list[[1]])))
all.tfs <- all.tfs[!is.na(all.tfs) & nzchar(all.tfs)]

if (sum(all.tfs %in% rownames(expr.mat)) < sum(all.tfs %in% colnames(expr.mat))) {
  expr.mat <- t(expr.mat)
}

# Analogous to GRNBoost postprocessing:
# 1) long-format TF-target importances
# 2) correlation term (stored as cor.p for compatibility)
# 3) score = importance * sign(cor.p), scaled globally to [-1, 1]
all.res <-
  plyr::ldply(names(tigress.by.ct),
              .fun = function(ct) {
                mat <- tigress.by.ct[[ct]]
                mat$TF <- rownames(mat)

                edges <- tidyr::pivot_longer(mat,
                                             cols = -TF,
                                             names_to = "target",
                                             values_to = "importance")

                tfs.in.expr <- intersect(unique(edges$TF), rownames(expr.mat))
                if (length(tfs.in.expr) == 0) {
                  warning(paste("No TFs from TIGRESS output found in expression matrix for cell type:", ct))
                  edges$cor.p <- NA_real_
                  edges$cell.type <- ct
                  return(edges)
                }

                cr <- cor(x = as.matrix(t(expr.mat[tfs.in.expr, , drop = FALSE])),
                          y = as.matrix(t(expr.mat)),
                          use = "pairwise.complete.obs")
                cr <- reshape2::melt(cr)
                colnames(cr) <- c("TF", "target", "cor.p")

                edges <- merge(edges, cr, by = c("TF", "target"), all.x = TRUE)
                edges$cell.type <- ct
                edges
              })

all.res$score <- all.res$importance * sign(all.res$cor.p)
max.abs.score <- max(abs(all.res$score), na.rm = TRUE)
if (is.finite(max.abs.score) && max.abs.score > 0) {
  all.res$score <- all.res$score / max.abs.score
}

# Add p-values analogously to GRNBoost postprocessing
pval_cutoff <- pnorm(4, lower.tail = FALSE)

all.res <-
  ddply(all.res, .variables = c("cell.type", "TF"),
        .fun = function(df) {
          sd.imp <- sd(df$importance, na.rm = TRUE)
          if (!is.finite(sd.imp) || sd.imp == 0) {
            df$pval <- 1
          } else {
            df$pval <- pnorm(df$importance, mean = 0, sd = sd.imp, lower.tail = FALSE)
          }
          df$padj <- p.adjust(df$pval)
          df$significant <- df$pval < pval_cutoff
          df
        })

final.edges <- all.res[, c("cell.type", "TF", "target", "importance", "cor.p", "score", "pval", "padj", "significant")]

data.table::fwrite(final.edges, file.path(tigress_path, "postprocessed-tigress_all-tfs-celltypes.csv.gz"))
final.edges

In [ ]:
sc.data <- fread_with_rownames(sc.data.file)
sc.metadata <- fread_with_rownames(sc.metadata.file)

In [ ]:
score.viper <- function(predictions.tbl, sc.metadata, sc.data,
                        sc.cell.type.col = "celltype_jf", sc.genotype.col = "gRNA_perturb", sc.wt.status = "CTRL")
{
  #' Apply VIPER to compute single-cell transcription factor (TF) activities from predicted TF targets.
  #'
  #' @description Compute VIPER TF activity scores using predicted TF targets and wild type (WT) and TF knockout (KO) scRNA-seq
  #' data.
  #'
  #' @param predictions.tbl data.frame. A data.frame of cell type-specific TF targets, where each row corresponds to a
  #' cell type-specific TF target, which is described by columns TF, cell.type, target, and score. TF and target are
  #' gene symbols, appearing as row names in the scRNA-seq data in sc.data. score is the strength of the interaction
  #' and should be in [-1, +1], with positive values corresponding to a positively regulated (induced) target and
  #' negative values corresponding to a negatively regulated (repressed) target.
  #' @param sc.metadata data.frame. Metadata describing the single-cell data, where each row is a cell. Row names
  #' correspond to the cell id. The cell's type is provided in column sc.cell.type.col and the KO'ed TF is in
  #' column sc.genotype.col. WT cells have gRNA_perturb set to CTRL.
  #' @param sc.data data.frame. scRNA-seq data, with row names gene symbols and column names cell ids. The data
  #' are log normalized.
  #' @param sc.cell.type.col. character string providing the name of the column in sc.metadata holding the cell type.
  #' @param sc.genotype.col. character string providing the name of the column in sc.metadata holding WT vs TF KO status.
  #' @param sc.wt.status. character string providing the status of WT cells in the sc.genotype.col
  #' @return A data.frame with VIPER TF activity scores for each cell. Each row is a cell, with VIPER TF activity score in column score.
  #' The TF is in column TF. The cell's type is in column cell.type and its WT or TF KO status
  #' is in column condition (as WT or KO, respectively).

  run_viper_compat <- function(mat, network) {
    rv <- decoupleR::run_viper
    rv_formals <- names(formals(rv))

    args <- list(mat, network)

    if (".source" %in% rv_formals) args[[".source"]] <- "source"
    if (".target" %in% rv_formals) args[[".target"]] <- "target"
    if (".mor" %in% rv_formals) args[[".mor"]] <- "mor"
    if ("source" %in% rv_formals) args[["source"]] <- "source"
    if ("target" %in% rv_formals) args[["target"]] <- "target"
    if ("mor" %in% rv_formals) args[["mor"]] <- "mor"
    if ("pleiotropy" %in% rv_formals) args[["pleiotropy"]] <- FALSE
    if ("method" %in% rv_formals) args[["method"]] <- "none"
    if ("minsize" %in% rv_formals) args[["minsize"]] <- 0
    if ("verbose" %in% rv_formals) args[["verbose"]] <- FALSE

    do.call(rv, args)
  }

  plyr::ddply(predictions.tbl,
        .variables = c("TF", "cell.type"),
        .parallel = FALSE,
        .fun = function(df) {
          tf <- df[1, "TF"]
          ct <- df[1, "cell.type"]

          ko.flag <- (sc.metadata[, sc.genotype.col] == tf) & (sc.metadata[, sc.cell.type.col] == ct)
          wt.flag <- (sc.metadata[, sc.genotype.col] == sc.wt.status) & (sc.metadata[, sc.cell.type.col] == ct)

          if (sum(ko.flag) == 0 || sum(wt.flag) == 0) {
            warning(paste("Skipping TF:", tf, "Celltype:", ct, "due to no KO or WT cells found for either KO or WT conditions."))
            return(data.frame(score = numeric(), perturbation = character(), condition = character(), TF = character(), cell.type = character()))
          }

          ko.mat <- sc.data[, ko.flag, drop = FALSE]
          wt.mat <- sc.data[, wt.flag, drop = FALSE]

          if (ncol(ko.mat) == 0 || ncol(wt.mat) == 0 || nrow(ko.mat) == 0 || nrow(wt.mat) == 0) {
            warning(paste("Skipping TF:", tf, "Celltype:", ct, "due to empty KO or WT expression matrix (0 rows or 0 columns)."))
            return(data.frame(score = numeric(), perturbation = character(), condition = character(), TF = character(), cell.type = character()))
          }

          combined.mat <- cbind(ko.mat, wt.mat)

          if (ncol(combined.mat) == 0 || nrow(combined.mat) == 0) {
            warning(paste("Skipping TF:", tf, "Celltype:", ct, "due to empty combined expression matrix."))
            return(data.frame(score = numeric(), perturbation = character(), condition = character(), TF = character(), cell.type = character()))
          }

          current_tf_targets_df <- df[df$TF == tf, c("target", "score"), drop = FALSE]

          if (nrow(current_tf_targets_df) == 0) {
            warning(paste("Skipping TF:", tf, "Celltype:", ct, "due to no targets found for TF in predictions.tbl."))
            return(data.frame(score = numeric(), perturbation = character(), condition = character(), TF = character(), cell.type = character()))
          }

          targets_in_expression <- intersect(current_tf_targets_df$target, rownames(combined.mat))
          if (length(targets_in_expression) == 0) {
            warning(paste("Skipping TF:", tf, "Celltype:", ct, "due to no targets found in expression data."))
            return(data.frame(score = numeric(), perturbation = character(), condition = character(), TF = character(), cell.type = character()))
          }

          filtered_tf_targets <- current_tf_targets_df[current_tf_targets_df$target %in% targets_in_expression, , drop = FALSE]
          if (nrow(filtered_tf_targets) == 0) {
            warning(paste("Skipping TF:", tf, "Celltype:", ct, "due to no filtered targets after intersect."))
            return(data.frame(score = numeric(), perturbation = character(), condition = character(), TF = character(), cell.type = character()))
          }

          network <- data.frame(
            source = as.character(tf),
            target = as.character(filtered_tf_targets$target),
            mor = filtered_tf_targets$score,
            stringsAsFactors = FALSE
          )

          # VIPER can be brittle with a single regulator, so add a dummy one.
          dummy.network <- network
          dummy.network$source <- "dummy"
          network <- rbind(network, dummy.network)

          vp <- tryCatch({
            run_viper_compat(combined.mat, network)
          }, error = function(e) {
            warning(paste("run_viper failed for TF:", tf, "Celltype:", ct, "Error:", e$message))
            return(NULL)
          })

          if (is.null(vp)) {
            return(data.frame(score = numeric(), perturbation = character(), condition = character(), TF = character(), cell.type = character()))
          }

          vp <- subset(vp, source != "dummy")
          vp <- merge(vp, sc.metadata[ko.flag | wt.flag, c(sc.genotype.col), drop = FALSE],
                      by.x = c("condition"), by.y = c("row.names"))
          vp <- vp[, !(colnames(vp) %in% c("condition", "statistic", "source", "p_value")), drop = FALSE]
          colnames(vp)[colnames(vp) == sc.genotype.col] <- "perturbation"
          vp$condition <- vp$perturbation
          vp$perturbation[vp$perturbation == sc.wt.status] <- "none"
          vp$condition[vp$condition != sc.wt.status] <- "KO"
          vp$condition[vp$condition == sc.wt.status] <- "WT"
          vp <- cbind(vp, TF = tf, cell.type = ct)
          vp
        })
}

In [ ]:
tmp <- tf.list[, c("TF", "celltype_jf")]
colnames(tmp) <- c("TF", "cell.type")
tmp <- merge(final.edges, tmp)
tg.preds <- subset(tmp, significant == TRUE)

In [ ]:
if (!requireNamespace("BiocManager", quietly = TRUE))
    install.packages("BiocManager")
BiocManager::install(c("dorothea", "decoupleR"))

In [ ]:
# Explicitly install and load the 'viper' package, which is a dependency for run_viper
BiocManager::install("viper", update = FALSE, ask = FALSE, force = TRUE)
library(dorothea)
library (decoupleR)
library(viper)

In [ ]:
# Compute VIPER TF activity scores for predictions using WT and TF KO scRNA-seq data

# tg.viper <- score.viper(tg.preds, sc.metadata, sc.data)
tg.viper <- score.viper(tg.preds, sc.metadata, sc.data)

In [ ]:
tg.viper.pvals <- compare.viper.distributions(tg.viper)
print(tg.viper.pvals)

In [ ]:
# Create a plot of the distributions of TF activity scores, stratified by WT or KO condition.
# g <- ggplot(data = tg.viper, aes(x = condition, y = score)) + geom_violin()
# Use ggpubr to include stats on plot
g <- ggviolin(data = tg.viper, "condition", "score") + stat_compare_means(method.args = list(alternative = "greater"), ref.group = "KO")
g <- g + facet_wrap(cell.type ~ TF, scales = "free_y")

In [ ]:
ofile <- paste0(plot.dir, "tigress-CHOOSE-TF-viper-distributions.png")
png(ofile)
print(g)
d <- dev.off()
print(ofile)